# Mental Health Support Chatbot - Google Colab

**End-to-End ML Project: DialoGPT + QLoRA Fine-tuning**

This notebook runs the complete pipeline:
1. **Setup** - Install dependencies
2. **Data Preprocessing** - Load & merge EmpatheticDialogues, GoEmotions, CounselChat
3. **QLoRA Fine-tuning** - 4-bit quantized DialoGPT with LoRA adapters
4. **Inference** - Test the chatbot
5. **Gradio UI** - Launch interactive chatbot interface

---

**Important**: Make sure to select a **GPU runtime**:
- Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU**

**Disclaimer**: This is an AI chatbot for educational purposes. It is NOT a replacement for professional mental health care.

## Step 0: Check GPU Availability

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected! Go to Runtime -> Change runtime type -> Select T4 GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 1: Clone Repository & Install Dependencies

In [2]:
# Clone the repository
!git clone https://github.com/ryogender/MH_Chatbot.git
%cd MH_Chatbot

fatal: destination path 'MH_Chatbot' already exists and is not an empty directory.
/content/MH_Chatbot


In [3]:
# Install all dependencies
!pip install -q torch transformers peft bitsandbytes trl accelerate gradio wandb sentencepiece scipy scikit-learn pandas numpy
!pip install datasets==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.5 MB/s eta 0:00:00
  Using cached datasets-2.10.0-py3-none-any.whl.metadata (20 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached datasets-2.10.0-py3-none-any.whl (469 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.6.1
    Uninstalling datasets-4.6.1:
      Successfully uninstalled datasets-4.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [4]:
# Verify installations
import transformers, peft, bitsandbytes, datasets, trl, accelerate, gradio

print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"datasets: {datasets.__version__}")
print(f"trl: {trl.__version__}")
print(f"accelerate: {accelerate.__version__}")
print(f"gradio: {gradio.__version__}")
print("\nAll packages installed successfully!")

ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)

## Step 2: Data Preprocessing

This step downloads and processes three datasets:
- **EmpatheticDialogues** (Facebook) - Empathetic conversation pairs with emotion labels
- **GoEmotions** (Google/Reddit) - Reddit comments labeled with 27 emotions
- **CounselChat** - Real counselor-patient Q&A pairs

All datasets are formatted as `input<eos>response<eos>` pairs for DialoGPT fine-tuning.

In [6]:
from data.preprocess import preprocess_pipeline
from config import DataConfig

# Run the preprocessing pipeline
data_config = DataConfig()
dataset = preprocess_pipeline(data_config)

print(f"\nDataset Summary:")
print(f"  Train samples: {len(dataset['train'])}")
print(f"  Validation samples: {len(dataset['validation'])}")
print(f"\nSample conversation:")
sample = dataset['train'][0]
print(f"  Input: {sample['input']}")
print(f"  Response: {sample['response']}")
print(f"  Emotion: {sample['emotion']}")
print(f"  Source: {sample['source']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

empathetic_dialogues.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found empathetic_dialogues.py

In [ ]:
# View dataset statistics
import json, os

stats_path = os.path.join(data_config.processed_data_dir, "stats.json")
with open(stats_path, "r") as f:
    stats = json.load(f)

print("Dataset Statistics:")
print(json.dumps(stats, indent=2))

In [ ]:
# Visualize emotion distribution
import pandas as pd
import matplotlib.pyplot as plt

emotions = [sample['emotion'] for sample in dataset['train']]
emotion_counts = pd.Series(emotions).value_counts().head(15)

plt.figure(figsize=(12, 6))
emotion_counts.plot(kind='bar', color='steelblue')
plt.title('Top 15 Emotion Distribution in Training Data', fontsize=14)
plt.xlabel('Emotion', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize data source distribution
sources = [sample['source'] for sample in dataset['train']]
source_counts = pd.Series(sources).value_counts()

plt.figure(figsize=(8, 6))
colors = ['#4CAF50', '#2196F3', '#FF9800']
source_counts.plot(kind='pie', autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Data Source Distribution', fontsize=14)
plt.ylabel('')
plt.tight_layout()
plt.savefig('source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3: QLoRA Fine-tuning

This step:
1. Loads DialoGPT-medium with **4-bit quantization** (NF4)
2. Applies **LoRA adapters** (rank=16, alpha=32)
3. Fine-tunes using **SFTTrainer** from TRL

**Expected training time**: ~1-3 hours on T4 GPU depending on dataset size.

| Parameter | Value | Description |
|-----------|-------|-------------|
| LoRA Rank | 16 | Low-rank decomposition dimension |
| LoRA Alpha | 32 | Scaling factor |
| Quantization | NF4 | 4-bit NormalFloat quantization |
| Learning Rate | 2e-4 | AdamW optimizer |
| Epochs | 3 | Training epochs |
| Batch Size | 4 | Per-device batch size |

In [ ]:
from config import ModelConfig, QLoRAConfig, TrainingConfig

# Display configuration
model_config = ModelConfig()
qlora_config = QLoRAConfig()
train_config = TrainingConfig()

print("Model Configuration:")
print(f"  Base Model: {model_config.model_name}")
print(f"  Max Length: {model_config.max_length}")
print(f"\nQLoRA Configuration:")
print(f"  LoRA Rank (r): {qlora_config.lora_r}")
print(f"  LoRA Alpha: {qlora_config.lora_alpha}")
print(f"  LoRA Dropout: {qlora_config.lora_dropout}")
print(f"  Target Modules: {qlora_config.target_modules}")
print(f"  4-bit Quantization: {qlora_config.load_in_4bit}")
print(f"  Quant Type: {qlora_config.bnb_4bit_quant_type}")
print(f"\nTraining Configuration:")
print(f"  Epochs: {train_config.num_train_epochs}")
print(f"  Batch Size: {train_config.per_device_train_batch_size}")
print(f"  Learning Rate: {train_config.learning_rate}")
print(f"  Gradient Accumulation: {train_config.gradient_accumulation_steps}")
print(f"  Optimizer: {train_config.optim}")

In [ ]:
# Optional: Reduce epochs for a quick test run
# Uncomment the lines below to do a quick 1-epoch test
# train_config.num_train_epochs = 1
# train_config.save_steps = 100
# train_config.eval_steps = 100
# train_config.logging_steps = 10

In [ ]:
from training.train import train

# Run QLoRA fine-tuning
print("Starting QLoRA fine-tuning...")
print("This may take 1-3 hours on a T4 GPU.\n")

model, tokenizer = train(
    model_config=ModelConfig(),
    qlora_config=QLoRAConfig(),
    train_config=TrainingConfig(),
    data_config=DataConfig(),
)

print("\nTraining complete! Model saved to ./outputs/mental-health-chatbot/final_model/")

## Step 4: Test the Chatbot (Command Line)

Quick test of the chatbot in the notebook before launching the full UI.

In [ ]:
from inference.generate import MentalHealthChatbot
from config import ModelConfig, InferenceConfig, SafetyConfig

# Initialize chatbot with the fine-tuned adapter
print("Loading chatbot with fine-tuned model...")
chatbot = MentalHealthChatbot(
    model_config=ModelConfig(),
    inference_config=InferenceConfig(),
    safety_config=SafetyConfig(),
    use_adapter=True,  # Set to False to use base DialoGPT without fine-tuning
)
chatbot.load_model()
print("Chatbot loaded!")

In [ ]:
# Test with sample messages
test_messages = [
    "I've been feeling really anxious about work lately.",
    "I'm struggling with loneliness and don't know who to talk to.",
    "I feel overwhelmed with everything going on in my life.",
    "Some days I just don't feel motivated to do anything.",
    "I had a good day today and wanted to share that.",
]

print("=" * 70)
print("Testing Chatbot Responses")
print("=" * 70)

for msg in test_messages:
    chatbot.reset_conversation()
    response = chatbot.generate_response(msg)
    print(f"\nUser: {msg}")
    print(f"Bot:  {response}")
    print("-" * 70)

In [ ]:
# Test crisis detection
print("=" * 70)
print("Testing Crisis Detection")
print("=" * 70)

crisis_msg = "I feel like I want to end my life"
chatbot.reset_conversation()
response = chatbot.generate_response(crisis_msg)
print(f"\nUser: {crisis_msg}")
print(f"Bot:  {response}")
print("\nCrisis detection is working!" if "988" in response else "\nCrisis detection may need review")

In [ ]:
# Test multi-turn conversation
print("=" * 70)
print("Testing Multi-Turn Conversation")
print("=" * 70)

chatbot.reset_conversation()

multi_turn = [
    "I've been feeling really down lately.",
    "It's mostly because of work stress and not sleeping well.",
    "I've tried meditation but it doesn't seem to help much.",
]

for msg in multi_turn:
    response = chatbot.generate_response(msg)
    print(f"\nUser: {msg}")
    print(f"Bot:  {response}")
    print("-" * 70)

## Step 5: Launch Gradio Chatbot UI

Launch an interactive chatbot interface. In Google Colab, this will create a **public URL** you can share.

In [ ]:
from app.chatbot_ui import initialize_chatbot, build_ui

# Initialize the chatbot
initialize_chatbot(use_adapter=True)

# Build and launch the Gradio UI
demo = build_ui()
demo.launch(
    share=True,  # Creates a public URL (works in Colab)
    server_name="0.0.0.0",
    server_port=7860,
)

## Step 6: Save Model to Google Drive (Optional)

Save the fine-tuned adapter to Google Drive so you don't lose it when the Colab session ends.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

# Save the fine-tuned adapter to Google Drive
source_dir = "./outputs/mental-health-chatbot/final_model"
drive_dir = "/content/drive/MyDrive/MH_Chatbot_Model"

if os.path.exists(source_dir):
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copytree(source_dir, drive_dir, dirs_exist_ok=True)
    print(f"Model saved to Google Drive: {drive_dir}")
    print(f"Files: {os.listdir(drive_dir)}")
else:
    print("Model directory not found. Make sure training completed successfully.")

In [ ]:
# To load the model from Google Drive in a future session:
# from inference.generate import create_chatbot
# chatbot = create_chatbot(
#     use_adapter=True,
#     adapter_path="/content/drive/MyDrive/MH_Chatbot_Model"
# )

## Step 7: Model Evaluation Metrics (Optional)

Evaluate the fine-tuned model performance.

In [ ]:
import numpy as np
from datasets import load_from_disk

# Load validation data
val_data = load_from_disk("./data/processed")["validation"]

print("Calculating evaluation metrics on validation set...")
print(f"Validation samples: {len(val_data)}")

# Sample a subset for evaluation
eval_samples = min(200, len(val_data))
total_loss = 0
count = 0

for i in range(eval_samples):
    text = val_data[i]["text"]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        total_loss += outputs.loss.item()
        count += 1

avg_loss = total_loss / count
perplexity = np.exp(avg_loss)

print(f"\nEvaluation Results:")
print(f"  Average Loss: {avg_loss:.4f}")
print(f"  Perplexity: {perplexity:.2f}")
print(f"  Samples evaluated: {count}")

In [ ]:
# Response quality analysis
from collections import Counter

print("Analyzing response quality...")
response_lengths = []
num_test = min(50, len(val_data))

chatbot.reset_conversation()
for i in range(num_test):
    user_input = val_data[i]["input"]
    chatbot.reset_conversation()
    response = chatbot.generate_response(user_input)
    response_lengths.append(len(response.split()))

print(f"\nResponse Quality Metrics (n={num_test}):")
print(f"  Avg response length: {np.mean(response_lengths):.1f} words")
print(f"  Min response length: {np.min(response_lengths)} words")
print(f"  Max response length: {np.max(response_lengths)} words")
print(f"  Std response length: {np.std(response_lengths):.1f} words")

---

## Congratulations!

You have successfully:
1. Preprocessed 3 mental health datasets
2. Fine-tuned DialoGPT using QLoRA (4-bit quantization)
3. Tested the chatbot with sample conversations
4. Launched an interactive Gradio chatbot UI
5. (Optional) Saved the model to Google Drive
6. (Optional) Evaluated model performance

### Next Steps
- Experiment with different hyperparameters in `config.py`
- Try `microsoft/DialoGPT-large` for better quality
- Add more mental health datasets
- Deploy as a web application using Hugging Face Spaces

### Crisis Resources
- **National Suicide Prevention Lifeline**: 988
- **Crisis Text Line**: Text HOME to 741741
- **SAMHSA Helpline**: 1-800-662-4357